# Does the dissociation occur without our intervention?

The paper's central measurement claim — that contribution magnitude and causal
necessity come apart — is currently demonstrated only in models we steered. The
objection that survives a rebuttal is that the effect is manufactured: we
penalise a route's contribution, so of course its contribution stops tracking
its necessity.

This notebook tests the claim in a pretrained model nobody steered.

**Model.** GPT-2 small, downloaded from HuggingFace, unmodified.

**Task.** Indirect-object identification, with the standard logit-difference
metric. Chosen deliberately: it is the setting in which direct logit attribution
and attribution patching were developed, the head structure is documented, and a
reviewer can rerun this in ten minutes. A result here is checkable in a way a
bespoke hybrid is not.

**Measures.** The same three as in the controlled experiments, applied to each
attention head's output: resample ablation as the ground truth, direct logit
attribution, and attribution patching.

**The prediction.** If the dissociation is real rather than manufactured, some
heads should be necessary by ablation while ranking low by contribution —
present and load-bearing, yet invisible to a magnitude proxy. That is the same
signature as the silencing result, in a model we did not touch.

An honest null is a usable outcome. If both proxies track the ablation closely
here, the claim scopes to regimes where a route is under active pressure, and
the paper says so. What is not usable is leaving the objection unanswered.

Runtime: about ten minutes on a T4, and it runs on CPU in roughly forty. Set
`MODEL_NAME` to `gpt2-medium` to check the result survives a size change.


In [ ]:
# !pip install -q transformers torch numpy pandas matplotlib scipy

import itertools, json, math, os, pathlib, random, time
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from scipy.stats import spearmanr

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ============================== CHOOSE HERE ==============================
MODEL_NAME = "gpt2"        # "gpt2" | "gpt2-medium" | "EleutherAI/pythia-160m"
N_BATCH    = 8             # evaluation batches per measurement
BATCH      = 32
WORKDIR    = "/content/wild"
# =========================================================================
#
# The paper's measurement claim is currently made only in models we steered.
# The obvious objection is that the effect is manufactured by the intervention.
# This notebook tests the claim in a pretrained model nobody steered, on a task
# with a documented circuit, using the two attribution methods practitioners
# actually use.
#
# GPT-2 small on indirect-object identification is chosen deliberately: it is
# the setting in which direct logit attribution and attribution patching were
# developed, the head structure is documented, and any reviewer can rerun it in
# ten minutes. A positive result here is checkable in a way a bespoke hybrid is
# not.

WORK = pathlib.Path(WORKDIR); WORK.mkdir(parents=True, exist_ok=True)
torch.set_grad_enabled(False)
print(f"device {DEVICE} | model {MODEL_NAME}")


def set_seed(s):
    torch.manual_seed(s); np.random.seed(s); random.seed(s)


set_seed(0)


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(DEVICE).eval()
cfg = model.config
N_LAYER = cfg.n_layer if hasattr(cfg, "n_layer") else cfg.num_hidden_layers
N_HEAD = cfg.n_head if hasattr(cfg, "n_head") else cfg.num_attention_heads
D_MODEL = cfg.n_embd if hasattr(cfg, "n_embd") else cfg.hidden_size
D_HEAD = D_MODEL // N_HEAD
print(f"{N_LAYER} layers x {N_HEAD} heads, d_model {D_MODEL}, d_head {D_HEAD}")
print(f"{sum(p.numel() for p in model.parameters())/1e6:.0f}M parameters")


def blocks():
    return model.transformer.h if hasattr(model, "transformer") else model.gpt_neox.layers


def attn_out_module(i):
    """The projection whose INPUT is the concatenation of per-head outputs. A
    pre-hook there can replace one head's slice without touching the others,
    which is what makes head-level ablation and attribution possible without a
    custom architecture."""
    b = blocks()[i]
    return b.attn.c_proj if hasattr(b, "attn") else b.attention.dense


# ---------------------------------------------------------------------------
# Indirect-object identification. "When Mary and John went to the store, John
# gave a drink to ___" — the answer is Mary. The metric is the logit difference
# between the indirect object and the subject at the final position, which is
# the standard IOI metric and is signed in a way that makes attribution
# meaningful.
# ---------------------------------------------------------------------------
NAMES = ["John", "Mary", "Tom", "James", "Dan", "Sarah", "Anna", "Paul",
         "Mark", "Kate", "Emma", "David", "Laura", "Peter", "Alice", "Robert",
         "Susan", "Michael", "Linda", "Kevin", "Julia", "Brian", "Rachel"]
NAMES = [n for n in NAMES if len(tok.encode(" " + n)) == 1]
print(f"{len(NAMES)} single-token names usable")

TEMPLATES = [
    "When {A} and {B} went to the {PLACE}, {B} gave a {OBJ} to",
    "Then {A} and {B} went to the {PLACE}. {B} gave a {OBJ} to",
    "After {A} and {B} went to the {PLACE}, {B} handed a {OBJ} to",
    "While {A} and {B} were at the {PLACE}, {B} passed a {OBJ} to",
]
PLACES = ["store", "park", "school", "office", "garden", "station"]
OBJS = ["drink", "book", "ball", "letter", "bag", "pen"]


def ioi_batch(n=BATCH, seed=None):
    """Each prompt names two people; the second (the subject) performs the
    action, so the answer is the first (the indirect object). Both names are in
    the context, so a model that has not solved the task cannot win by
    frequency alone."""
    rng = random.Random(seed)
    prompts, io_ids, s_ids = [], [], []
    for _ in range(n):
        a, b = rng.sample(NAMES, 2)
        t = rng.choice(TEMPLATES)
        prompts.append(t.format(A=a, B=b, PLACE=rng.choice(PLACES),
                                OBJ=rng.choice(OBJS)))
        io_ids.append(tok.encode(" " + a)[0])
        s_ids.append(tok.encode(" " + b)[0])
    enc = tok(prompts, return_tensors="pt", padding=True) if tok.pad_token \
        else tok(prompts, return_tensors="pt")
    return {"input_ids": enc["input_ids"].to(DEVICE),
            "io": torch.tensor(io_ids, device=DEVICE),
            "s": torch.tensor(s_ids, device=DEVICE)}


def logit_diff(logits, b):
    """logit(indirect object) - logit(subject) at the final position."""
    last = logits[:, -1, :]
    return (last.gather(1, b["io"][:, None]) - last.gather(1, b["s"][:, None])).squeeze(1)


# sanity: does the model actually do the task?
_accs, _lds = [], []
for i in range(4):
    b = ioi_batch(seed=i)
    lg = model(b["input_ids"]).logits
    ld = logit_diff(lg, b)
    _lds.append(ld.mean().item())
    _accs.append((ld > 0).float().mean().item())
print(f"\nIOI: mean logit difference {np.mean(_lds):+.3f}, "
      f"correct on {np.mean(_accs):.1%} of prompts")
if np.mean(_accs) < 0.8:
    print("  !! the model is not solving the task; the attribution comparison")
    print("     below would be measuring noise. Try a larger model.")


In [ ]:
# ---------------------------------------------------------------------------
# Three measures of "does this head matter", on the same model and the same
# prompts. All three operate on the head's output z, the slice of the
# concatenated per-head activations that feeds the output projection.
#
#   RESAMPLE ABLATION  replace z with the same head's z from a shuffled batch
#                      and measure the logit difference lost. The ground truth,
#                      and the intervention the other two approximate.
#   DIRECT LOGIT ATTR  the head's own write to the logit difference, obtained by
#                      pushing z through W_O and onto the unembedding direction.
#   ATTRIBUTION PATCH  (z_clean - z_corrupt) . dM/dz, the standard linear
#                      approximation to the ablation, one backward pass for all
#                      heads at once.
#
# The paper's claim, transposed to this setting: the two cheap measures should
# systematically underrate heads that the ablation says are necessary.
# ---------------------------------------------------------------------------

def _slice(h):
    return slice(h * D_HEAD, (h + 1) * D_HEAD)


def run_with_head_patch(ids, layer, head, source):
    """Forward pass with one head's output replaced by `source`."""
    mod = attn_out_module(layer)

    def pre(m, args):
        x = args[0].clone()
        x[:, :, _slice(head)] = source
        return (x,)

    hk = mod.register_forward_pre_hook(pre)
    try:
        return model(ids).logits
    finally:
        hk.remove()


def capture_head_inputs(ids):
    """The concatenated per-head activations entering each output projection."""
    store = {}
    hooks = [attn_out_module(i).register_forward_pre_hook(
        lambda m, args, i=i: store.__setitem__(i, args[0].detach())) 
        for i in range(N_LAYER)]
    try:
        logits = model(ids).logits
    finally:
        for h in hooks:
            h.remove()
    return logits, store


def resample_ablation(n_batch=N_BATCH):
    """Ground truth. For each head, replace its output with the same head's
    output on a shuffled copy of the batch — an in-distribution activation that
    carries no information about this prompt — and record the logit difference
    lost, as a fraction of the clean logit difference."""
    eff = np.zeros((N_LAYER, N_HEAD))
    for bi in range(n_batch):
        b = ioi_batch(seed=1000 + bi)
        clean_logits, acts = capture_head_inputs(b["input_ids"])
        base = logit_diff(clean_logits, b).mean().item()
        perm = torch.randperm(b["input_ids"].shape[0], device=DEVICE)
        for L in range(N_LAYER):
            src_all = acts[L][perm]
            for H in range(N_HEAD):
                lg = run_with_head_patch(b["input_ids"], L, H,
                                         src_all[:, :, _slice(H)])
                eff[L, H] += (base - logit_diff(lg, b).mean().item()) / abs(base)
        if bi == 0:
            print(f"  clean logit difference {base:+.3f}")
    return eff / n_batch


def direct_logit_attribution(n_batch=N_BATCH):
    """Each head's own contribution to the logit difference: its output pushed
    through the output projection and read off in the unembedding direction of
    (indirect object - subject). The final layer norm is applied as a scale
    taken from the clean run, which is the standard treatment."""
    W_U = model.get_output_embeddings().weight               # (vocab, d_model)
    ln_f = model.transformer.ln_f if hasattr(model, "transformer") \
        else model.gpt_neox.final_layer_norm
    dla = np.zeros((N_LAYER, N_HEAD))
    for bi in range(n_batch):
        b = ioi_batch(seed=2000 + bi)
        _, acts = capture_head_inputs(b["input_ids"])
        # the residual-stream scale the final layer norm applies at the last
        # position, so head contributions are compared in logit units
        resid = []
        hk = ln_f.register_forward_pre_hook(
            lambda m, args: resid.append(args[0].detach()))
        model(b["input_ids"]); hk.remove()
        scale = (resid[0][:, -1, :].var(-1, keepdim=True) + ln_f.eps).sqrt()
        d = (W_U[b["io"]] - W_U[b["s"]])                     # (B, d_model)
        for L in range(N_LAYER):
            W_O = attn_out_module(L).weight                  # Conv1D: (in, out)
            if W_O.shape[0] != D_MODEL:
                W_O = W_O.t()
            z = acts[L][:, -1, :]
            for H in range(N_HEAD):
                contrib = z[:, _slice(H)] @ W_O[_slice(H), :]
                dla[L, H] += ((contrib / scale) * d).sum(-1).mean().item()
    return dla / n_batch


def attribution_patching(n_batch=N_BATCH):
    """(z_clean - z_corrupt) . dM/dz, with the same shuffled corruption the
    ablation uses, so the two measure the same intervention — one exactly, one
    to first order."""
    attr = np.zeros((N_LAYER, N_HEAD))
    with torch.enable_grad():
        for bi in range(n_batch):
            b = ioi_batch(seed=3000 + bi)
            store, hooks = {}, []
            for i in range(N_LAYER):
                def pre(m, args, i=i):
                    x = args[0].clone().requires_grad_(True)
                    x.retain_grad(); store[i] = x
                    return (x,)
                hooks.append(attn_out_module(i).register_forward_pre_hook(pre))
            try:
                M = logit_diff(model(b["input_ids"]).logits, b).mean()
                model.zero_grad(set_to_none=True)
                M.backward()
            finally:
                for h in hooks:
                    h.remove()
            perm = torch.randperm(b["input_ids"].shape[0], device=DEVICE)
            for L in range(N_LAYER):
                z = store[L].detach(); g = store[L].grad
                if g is None:
                    continue
                diff = z - z[perm]
                for H in range(N_HEAD):
                    attr[L, H] += ((diff[:, :, _slice(H)]
                                    * g[:, :, _slice(H)]).sum(-1).sum(-1)
                                   ).mean().item()
    return attr / n_batch


In [ ]:
t0 = time.time()
print("resample ablation (the ground truth) — this is the slow one")
ABL = resample_ablation()
print(f"  {(time.time()-t0)/60:.1f}m\n")
print("direct logit attribution"); DLA = direct_logit_attribution()
print("attribution patching");     ATTR = attribution_patching()

R = pd.DataFrame([{"layer": L, "head": H, "ablation": ABL[L, H],
                   "dla": DLA[L, H], "attr_patch": ATTR[L, H]}
                  for L in range(N_LAYER) for H in range(N_HEAD)])
R.to_csv(WORK / f"heads_{MODEL_NAME.replace('/', '_')}.csv", index=False)
print(f"\n{len(R)} heads measured\n")

# ---- do the cheap measures rank heads the way the ablation does? ----------
print("agreement with resample ablation, over all heads:")
for c in ["dla", "attr_patch"]:
    rho = spearmanr(R.ablation, R[c]).correlation
    r = np.corrcoef(R.ablation, R[c])[0, 1]
    print(f"  {c:<12} Spearman {rho:+.3f}   Pearson {r:+.3f}")

# ---- the specific failure the paper predicts -----------------------------
# A head that the ablation says is necessary, while the contribution measure
# says it writes little: representationally present, functionally load-bearing,
# and invisible to a magnitude proxy. That is the silencing signature, in a
# model nobody intervened on.
print("\nheads the ablation ranks in the top decile:")
top = R.nlargest(max(len(R) // 10, 5), "ablation")
for c in ["dla", "attr_patch"]:
    R[f"rank_{c}"] = R[c].rank(ascending=False)
R["rank_abl"] = R.ablation.rank(ascending=False)
show = top.assign(**{f"rank_{c}": R.loc[top.index, f"rank_{c}"]
                     for c in ["dla", "attr_patch"]})
print(show[["layer", "head", "ablation", "dla", "attr_patch",
            "rank_dla", "rank_attr_patch"]].round(4).to_string(index=False))

MISSED = {}
for c in ["dla", "attr_patch"]:
    miss = top[R.loc[top.index, f"rank_{c}"] > len(R) / 2]
    MISSED[c] = miss
    print(f"\n  {c}: {len(miss)}/{len(top)} of the ablation's top-decile heads "
          f"are ranked in the BOTTOM HALF")
    if len(miss):
        print(miss[["layer", "head", "ablation", c]].round(4).to_string(index=False))

# ---- and the reverse error -----------------------------------------------
print("\nheads the contribution measures rank highest, and what ablation says:")
for c in ["dla", "attr_patch"]:
    t = R.nlargest(5, c)
    print(f"  {c}: mean ablation effect of its top 5 = {t.ablation.mean():.4f} "
          f"(top 5 by ablation: {R.nlargest(5, 'ablation').ablation.mean():.4f})")

print("\n" + "=" * 74)
worst = max(len(MISSED[c]) for c in MISSED)
if worst:
    print(f"  In {MODEL_NAME}, a model we did not train or steer, "
          f"{worst} of the {len(top)}")
    print("  heads most necessary to the task by resample-ablation are ranked in")
    print("  the bottom half by a contribution measure. The dissociation between")
    print("  contribution magnitude and causal necessity is therefore not an")
    print("  artefact of the intervention used in the controlled experiments —")
    print("  it occurs in a pretrained model on a documented circuit.")
else:
    print("  Both contribution measures track the ablation ranking closely here.")
    print("  Report that: the dissociation is not visible in this model on this")
    print("  task, so the controlled setting is where it can be studied and the")
    print("  claim must stay scoped to regimes where a route is under pressure.")
    print("  An honest null here is worth more than a strained positive.")
print("=" * 74)

# ---- figure ---------------------------------------------------------------
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(7.5, 3.2))
for a, c, name in [(ax[0], "dla", "direct logit attribution"),
                   (ax[1], "attr_patch", "attribution patching")]:
    a.scatter(R[c], R.ablation, s=14, alpha=.65, color="#8d99ae")
    m = R.loc[top.index]
    a.scatter(m[c], m.ablation, s=26, color="#d1495b", label="top decile by ablation")
    a.axhline(0, color="k", lw=.6); a.axvline(0, color="k", lw=.6)
    a.set_xlabel(name); a.set_ylabel("logit difference lost when resampled")
    a.set_title(f"Spearman {spearmanr(R.ablation, R[c]).correlation:+.2f}")
ax[0].legend(frameon=False, fontsize=7)
fig.suptitle(f"{MODEL_NAME} on IOI: contribution against necessity, "
             f"{len(R)} attention heads", fontsize=9)
fig.tight_layout()
for ext in ("pdf", "png"):
    fig.savefig(WORK / f"wild_{MODEL_NAME.replace('/', '_')}.{ext}", dpi=200,
                bbox_inches="tight")
plt.show()
print(f"\nsaved to {WORK}")
